In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

### cleaning data - misssing values and correlation

In [2]:
dataframe_check = pd.read_csv("../data/padel_loop_results_BBB.csv")
dataframe_check

,Original_Name,Original_SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,...,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds,BBB
0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,1,-1.6366,2.678460,20.9255,52.325102,18,18,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,4,-1.8532,3.434350,84.1596,65.253860,11,11,56,...,0.202821,0.515082,0.311531,0.404004,22.493883,107.951242,249.714723,0.589017,1.230617,0
2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,0,1.7041,2.903957,22.1264,28.605965,10,11,18,...,0.348308,0.488440,0.490672,0.207540,7.749546,14.229627,23.592414,0.476530,1.186652,0
3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,0,1.3081,1.711126,58.1882,43.238688,6,6,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,1,-2.3618,5.578099,88.3588,66.801411,6,6,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9579,licostinel,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl,0,0.1909,0.036443,57.1443,26.948379,6,6,20,...,0.350759,0.566435,0.535897,0.365741,8.796186,19.954489,35.953677,0.424423,1.468073,1
9580,ademetionine(adenosyl-methionine),[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...,0,-4.7530,22.591009,84.4263,54.579446,0,0,49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
9581,mesocarb,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...,0,1.4656,2.147983,98.9391,49.686274,0,0,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
9582,tofisoline,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...,0,-0.8870,0.786769,113.2376,61.464618,0,0,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [3]:
def load_and_inspect(filepath):
    """Load data and compute initial missingness statistics"""
    df = pd.read_csv(filepath)
    
    # Separate features and target
    X = df.drop(columns=['BBB'])
    y = df['BBB']
    
    print("="*70)
    print("STEP 1: INITIAL DATA INSPECTION")
    print("="*70)
    print(f"Dataset shape: {df.shape} (molecules × descriptors+target)")
    print(f"Target distribution:\n{y.value_counts()}\n")
    
    # Compute column-wise missingness
    col_missing_pct = (X.isnull().sum() / len(X) * 100).sort_values(ascending=False)
    col_missing_pct = col_missing_pct[col_missing_pct > 0]
    
    print(f"Descriptors with missing values: {len(col_missing_pct)}")
    if len(col_missing_pct) > 0:
        print(f"Top 10 most missing descriptors:")
        print(col_missing_pct.head(10))
    print()
    
    # Compute row-wise missingness
    row_missing_pct = (X.isnull().sum(axis=1) / X.shape[1] * 100)
    print(f"Row-wise missingness:")
    print(f"  Mean: {row_missing_pct.mean():.2f}%")
    print(f"  Max: {row_missing_pct.max():.2f}%")
    print(f"  Molecules with >10% missing: {(row_missing_pct > 10).sum()}\n")
    
    return X, y

# Load data
X, y = load_and_inspect("../data/padel_loop_results_BBB.csv")

STEP 1: INITIAL DATA INSPECTION
Dataset shape: (9584, 1878) (molecules × descriptors+target)
Target distribution:
BBB
1    6792
0    2792
Name: count, dtype: int64

Descriptors with missing values: 717
Top 10 most missing descriptors:
Ds     27.003339
Di     27.003339
E3u    27.003339
Du     27.003339
E3m    27.003339
Dm     27.003339
Dv     27.003339
E3e    27.003339
De     27.003339
E3p    27.003339
dtype: float64

Row-wise missingness:
  Mean: 6.26%
  Max: 33.24%
  Molecules with >10% missing: 2586



In [4]:
X

,Original_Name,Original_SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nAromBond,nAtom,...,P1s,P2s,E1s,E2s,E3s,Ts,As,Vs,Ks,Ds
0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,1,-1.6366,2.678460,20.9255,52.325102,18,18,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,4,-1.8532,3.434350,84.1596,65.253860,11,11,56,...,0.726012,0.202821,0.515082,0.311531,0.404004,22.493883,107.951242,249.714723,0.589017,1.230617
2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,0,1.7041,2.903957,22.1264,28.605965,10,11,18,...,0.636046,0.348308,0.488440,0.490672,0.207540,7.749546,14.229627,23.592414,0.476530,1.186652
3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,0,1.3081,1.711126,58.1882,43.238688,6,6,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,1,-2.3618,5.578099,88.3588,66.801411,6,6,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9579,licostinel,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl,0,0.1909,0.036443,57.1443,26.948379,6,6,20,...,0.598857,0.350759,0.566435,0.535897,0.365741,8.796186,19.954489,35.953677,0.424423,1.468073
9580,ademetionine(adenosyl-methionine),[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...,0,-4.7530,22.591009,84.4263,54.579446,0,0,49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9581,mesocarb,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...,0,1.4656,2.147983,98.9391,49.686274,0,0,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9582,tofisoline,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...,0,-0.8870,0.786769,113.2376,61.464618,0,0,54,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def check_missing_value_descriptor_importance(X, y):
    """Check if descriptors with missing values have important correlation with target"""
    
    print("="*70)
    print("ANALYSIS: IMPORTANCE OF MISSING-VALUE DESCRIPTORS")
    print("="*70)
    
    # Select only numeric columns
    X_numeric = X.select_dtypes(include=[np.number])
    
    # Identify descriptors with any missing values (from numeric columns only)
    missing_val_cols = X_numeric.columns[X_numeric.isnull().sum() > 0].tolist()
    clean_cols = X_numeric.columns[X_numeric.isnull().sum() == 0].tolist()
    
    print(f"Descriptors with missing values: {len(missing_val_cols)}")
    print(f"Clean descriptors (no missing): {len(clean_cols)}\n")
    
    # Fill NaN with median for correlation computation
    X_filled = X_numeric.fillna(X_numeric.median())
    
    # Calculate correlation with target for missing-value descriptors
    correlations = X_filled[missing_val_cols].corrwith(y).abs().sort_values(ascending=False)
    
    # Summary statistics
    strong_corr = (correlations > 0.15).sum()
    moderate_corr = ((correlations > 0.10) & (correlations <= 0.15)).sum()
    
    print(f"Correlation with BBB target:")
    print(f"  Strong (>0.15): {strong_corr}")
    print(f"  Moderate (0.10-0.15): {moderate_corr}")
    print(f"  Weak (≤0.10): {len(correlations) - strong_corr - moderate_corr}\n")
    
    # Show top 15 most important missing-value descriptors
    print(f"Top 15 most important missing-value descriptors (by |correlation|):")
    print(correlations.head(15))
    print()
    
    return correlations

# Analyze missing-value descriptor importance BEFORE filtering
missing_corr = check_missing_value_descriptor_importance(X, y)

ANALYSIS: IMPORTANCE OF MISSING-VALUE DESCRIPTORS
Descriptors with missing values: 716
Clean descriptors (no missing): 1159

Correlation with BBB target:
  Strong (>0.15): 207
  Moderate (0.10-0.15): 231
  Weak (≤0.10): 278

Top 15 most important missing-value descriptors (by |correlation|):
SM1_Dzv       0.346061
SM1_Dzi       0.331053
SM1_DzZ       0.327008
SM1_Dze       0.325242
SM1_Dzm       0.321718
SM1_Dzp       0.307761
ETA_EtaP_F    0.291884
ATSC0s        0.290628
SpMax7_Bhs    0.286155
ETA_Eta_F     0.285209
SpMax6_Bhs    0.284859
SM1_Dzs       0.280531
SpMax5_Bhs    0.275605
SpMax8_Bhs    0.275230
TPSA          0.274428
dtype: float64



In [6]:
def filter_high_missingness_descriptors(X, missing_threshold=30):
    """Remove descriptors with missingness exceeding threshold"""
    missing_pct = (X.isnull().sum() / len(X) * 100)
    cols_to_keep = missing_pct[missing_pct <= missing_threshold].index
    
    print("="*70)
    print("STEP 2: FILTER HIGH-MISSINGNESS DESCRIPTORS")
    print("="*70)
    print(f"Threshold: {missing_threshold}%")
    print(f"Descriptors before: {X.shape[1]}")
    print(f"Descriptors removed: {X.shape[1] - len(cols_to_keep)}")
    print(f"Descriptors after: {len(cols_to_keep)}\n")
    
    return X[cols_to_keep]

X = filter_high_missingness_descriptors(X, missing_threshold=15)

STEP 2: FILTER HIGH-MISSINGNESS DESCRIPTORS
Threshold: 15%
Descriptors before: 1877
Descriptors removed: 431
Descriptors after: 1446



In [ ]:
def remove_correlated_features(X, corr_threshold=0.95):
    """Remove redundant features based on correlation; keep one from each correlated pair"""
    
    print("="*70)
    print("STEP 3: REMOVE CORRELATED DESCRIPTORS")
    print("="*70)
    
    # Fill NaN with median for correlation computation (numeric columns only)
    X_numeric = X.select_dtypes(include=[np.number])
    X_filled = X_numeric.fillna(X_numeric.median())
    corr_matrix = X_filled.corr().abs()
    
    # Identify correlated pairs (upper triangle)
    upper_triangle = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    
    # Find correlated feature pairs
    to_drop = set()
    for column in upper_triangle.columns:
        corr_cols = upper_triangle[column][upper_triangle[column] > corr_threshold].index
        for corr_col in corr_cols:
            # Keep feature with lower missingness
            miss_col = X_numeric[column].isnull().sum()
            miss_corr_col = X_numeric[corr_col].isnull().sum()
            if miss_col >= miss_corr_col:
                to_drop.add(column)
            else:
                to_drop.add(corr_col)
    
    X_filtered = X_numeric.drop(columns=list(to_drop))
    
    print(f"Correlation threshold: >{corr_threshold}")
    print(f"Descriptors before: {X_numeric.shape[1]}")
    print(f"Descriptors removed: {len(to_drop)}")
    print(f"Descriptors after: {X_filtered.shape[1]}\n")
    
    return X_filtered

X = remove_correlated_features(X, corr_threshold=0.95)

STEP 3: REMOVE CORRELATED DESCRIPTORS
Correlation threshold: >0.95
Descriptors before: 1444
Descriptors removed: 511
Descriptors after: 933



In [8]:
def filter_high_missingness_molecules(X, y, missing_threshold=15):
    """Remove molecules with missingness exceeding threshold"""
    missing_pct = (X.isnull().sum(axis=1) / X.shape[1] * 100)
    mask = missing_pct <= missing_threshold
    
    print("="*70)
    print("STEP 4: FILTER HIGH-MISSINGNESS MOLECULES")
    print("="*70)
    print(f"Threshold: {missing_threshold}%")
    print(f"Molecules before: {len(X)}")
    print(f"Molecules removed: {(~mask).sum()}")
    print(f"Molecules after: {mask.sum()}\n")
    
    return X[mask], y[mask]

X, y = filter_high_missingness_molecules(X, y, missing_threshold=15)

STEP 4: FILTER HIGH-MISSINGNESS MOLECULES
Threshold: 15%
Molecules before: 9584
Molecules removed: 0
Molecules after: 9584



In [9]:
print("="*70)
print("MISSING VALUES CHECK (AFTER FILTERING)")
print("="*70)

total_missing = X.isnull().sum().sum()
rows_with_missing = (X.isnull().sum(axis=1) > 0).sum()
cols_with_missing = (X.isnull().sum() > 0).sum()

print(f"Dataset shape: {X.shape[0]} molecules × {X.shape[1]} descriptors")
print(f"Total missing values: {total_missing}")
print(f"Rows with ANY missing: {rows_with_missing}/{X.shape[0]} ({100*rows_with_missing/X.shape[0]:.1f}%)")
print(f"Columns with ANY missing: {cols_with_missing}/{X.shape[1]}")

if total_missing == 0:
    print("\n✓ No missing values - data is clean!")
else:
    print(f"\n⚠ Still have {total_missing} missing values to handle with imputation")
print("="*70)


MISSING VALUES CHECK (AFTER FILTERING)
Dataset shape: 9584 molecules × 933 descriptors
Total missing values: 5427
Rows with ANY missing: 121/9584 (1.3%)
Columns with ANY missing: 138/933

⚠ Still have 5427 missing values to handle with imputation


In [10]:
#choosing to drop molecules here as its a less significant chunk for molecules (1.3%) as compared to descriptors (14.7%)
# Drop rows with ANY missing values
X_clean = X.dropna(axis=0)


print("="*70)
print("DATA AFTER DROPPING MISSING VALUES")
print("="*70)
print(f"Shape: {X_clean.shape[0]} molecules × {X_clean.shape[1]} descriptors")
print(f"Missing values: {X_clean.isnull().sum().sum()}")
print()
print(type(X_clean))

DATA AFTER DROPPING MISSING VALUES
Shape: 9463 molecules × 933 descriptors
Missing values: 0

<class 'pandas.core.frame.DataFrame'>


In [11]:
#now that #molecules is different in X and y - have to make sure ordering is presevred and match accordingly
# Check if indices match
print(X_clean.index.equals(y.index))  # True if perfectly aligned
print(X_clean.shape[0] == y.shape[0])  # True if same length

False
False


In [12]:
y_clean = y.loc[X_clean.index]  # Realign by index
y_clean

0       0
1       0
2       0
3       0
4       0
       ..
9579    1
9580    1
9581    1
9582    1
9583    1
Name: BBB, Length: 9463, dtype: int64

In [13]:
# Reload original data
df_original = pd.read_csv("../data/padel_loop_results_BBB.csv")

# Get the SMILES and name columns for the molecules still in X_clean
molecules_metadata = df_original.loc[X_clean.index, ['Original_Name', 'Original_SMILES']]  # adjust column names as needed

# Combine with your clean features
X_final = pd.concat([molecules_metadata, X_clean], axis=1)

print(X_final.head())
print(X_final.shape)  # Should be (9463, 2 + 933) = (9463, 935)

                         Original_Name  \
0                       sulphasalazine   
1                           moxalactam   
2                           clioquinol   
3  bbcpd11 (cimetidine analog) (y-g13)   
4                        schembl614298   

                                     Original_SMILES  nAcid   ALogP    ALogp2  \
0   O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O      1 -1.6366  2.678460   
1  COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...      4 -1.8532  3.434350   
2                             Oc1c(I)cc(Cl)c2cccnc12      0  1.7041  2.903957   
3                         CCNC(=NCCSCc1ncccc1Br)NC#N      0  1.3081  1.711126   
4  CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...      1 -2.3618  5.578099   

       AMR       apol  naAromAtom  nB  nN  ...     VE2_D      VE3_D  \
0  20.9255  52.325102          18   0   4  ...  0.000796 -10.652046   
1  84.1596  65.253860          11   0   6  ...  0.001968  -9.530098   
2  22.1264  28.605965          10   0   1  ... 

In [14]:
X_final

,Original_Name,Original_SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nB,nN,...,VE2_D,VE3_D,VR1_D,VR2_D,VR3_D,TopoPSA,SRW5,AMW,WPATH,XLogP
0,sulphasalazine,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,1,-1.6366,2.678460,20.9255,52.325102,18,0,4,...,0.000796,-10.652046,303.624671,10.843738,16.004218,149.69,0.000000,9.477821,2428.0,4.845
1,moxalactam,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,4,-1.8532,3.434350,84.1596,65.253860,11,0,6,...,0.001968,-9.530098,667.876683,18.552130,23.414773,231.60,2.397895,9.287522,4114.0,-0.302
2,clioquinol,Oc1c(I)cc(Cl)c2cccnc12,0,1.7041,2.903957,22.1264,28.605965,10,0,1,...,0.001428,-5.182295,98.303672,7.561821,5.964480,33.12,0.000000,16.939469,218.0,2.584
3,bbcpd11 (cimetidine analog) (y-g13),CCNC(=NCCSCc1ncccc1Br)NC#N,0,1.3081,1.711126,58.1882,43.238688,6,0,5,...,0.002822,-5.559286,360.785620,18.988717,11.187739,98.40,0.000000,9.743742,898.0,2.970
4,schembl614298,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,1,-2.3618,5.578099,88.3588,66.801411,6,0,1,...,0.007298,-4.697970,463.869102,14.056639,20.260688,149.15,2.397895,7.686143,2850.0,-0.155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9579,licostinel,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl,0,0.1909,0.036443,57.1443,26.948379,6,0,3,...,0.003040,-5.036402,255.864815,15.050871,9.425904,101.34,0.000000,13.747503,460.0,2.533
9580,ademetionine(adenosyl-methionine),[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...,0,-4.7530,22.591009,84.4263,54.579446,0,0,6,...,0.009701,-3.617143,389.163321,14.413456,16.102798,182.18,3.044522,8.125250,1993.0,-4.423
9581,mesocarb,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...,0,1.4656,2.147983,98.9391,49.686274,0,0,4,...,0.001331,-8.264830,440.311114,18.346296,14.609956,44.70,2.397895,7.670071,1597.0,3.090
9582,tofisoline,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...,0,-0.8870,0.786769,113.2376,61.464618,0,0,2,...,0.005528,-5.223972,454.650380,16.237514,17.134680,39.93,0.000000,7.077579,1854.0,3.247


In [15]:
#sanity check - make sure the correct molecule names associate with correct BBB values - check with original data (dataframe check) 

# Create mapping from X_final: Original_Name -> BBB value
mapping_cleaned = pd.DataFrame({
    'Original_Name': X_final['Original_Name'],
    'BBB_cleaned': y_clean.values
}, index=X_final.index)

# Create mapping from original dataframe
mapping_original = pd.DataFrame({
    'Original_Name': dataframe_check['Original_Name'],
    'BBB_original': dataframe_check['BBB']
}, index=dataframe_check.index)

# Merge and compare
comparison = mapping_cleaned.merge(
    mapping_original.drop_duplicates(subset=['Original_Name']), 
    on='Original_Name', 
    how='left'
)

# Check for mismatches
mismatches = comparison[comparison['BBB_cleaned'] != comparison['BBB_original']]

print("="*70)
print("SANITY CHECK: MOLECULE NAME → BBB VALUE ALIGNMENT")
print("="*70)
print(f"\nSample mappings (first 10 rows):")
print(comparison[['Original_Name', 'BBB_cleaned', 'BBB_original']].head(10))
print(f"\nTotal molecules in cleaned data: {len(comparison)}")
print(f"Mismatches found: {len(mismatches)}")

if len(mismatches) == 0:
    print("✓ PASS - All BBB values are correctly aligned!")
else:
    print("✗ FAIL - Mismatches detected:")
    print(mismatches[['Original_Name', 'BBB_cleaned', 'BBB_original']])

print("="*70)


SANITY CHECK: MOLECULE NAME → BBB VALUE ALIGNMENT

Sample mappings (first 10 rows):
                         Original_Name  BBB_cleaned  BBB_original
0                       sulphasalazine            0             0
1                           moxalactam            0             0
2                           clioquinol            0             0
3  bbcpd11 (cimetidine analog) (y-g13)            0             0
4                        schembl614298            0             0
5                           uk-240,455            0             0
6               morphine-6-glucuronide            0             0
7                       nitrofurantoin            0             0
8                            l-701,324            0             0
9                           33419-42-0            0             0

Total molecules in cleaned data: 9463
Mismatches found: 0
✓ PASS - All BBB values are correctly aligned!


In [31]:
X_final
print(X_final["Original_Name"].isna().sum())
print(X_final["Original_SMILES"].isna().sum())

#having names in the X dataset is introducing missing values - all smiles present - droping names
X_final = X_final.drop(columns=['Original_Name'])
X_final

1097
0


,Original_SMILES,nAcid,ALogP,ALogp2,AMR,apol,naAromAtom,nB,nN,nO,...,VE2_D,VE3_D,VR1_D,VR2_D,VR3_D,TopoPSA,SRW5,AMW,WPATH,XLogP
0,O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O,1,-1.6366,2.678460,20.9255,52.325102,18,0,4,5,...,0.000796,-10.652046,303.624671,10.843738,16.004218,149.69,0.000000,9.477821,2428.0,4.845
1,COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...,4,-1.8532,3.434350,84.1596,65.253860,11,0,6,9,...,0.001968,-9.530098,667.876683,18.552130,23.414773,231.60,2.397895,9.287522,4114.0,-0.302
2,Oc1c(I)cc(Cl)c2cccnc12,0,1.7041,2.903957,22.1264,28.605965,10,0,1,1,...,0.001428,-5.182295,98.303672,7.561821,5.964480,33.12,0.000000,16.939469,218.0,2.584
3,CCNC(=NCCSCc1ncccc1Br)NC#N,0,1.3081,1.711126,58.1882,43.238688,6,0,5,0,...,0.002822,-5.559286,360.785620,18.988717,11.187739,98.40,0.000000,9.743742,898.0,2.970
4,CN1CC[C@]23c4c5ccc(OC6O[C@H](C(=O)O)[C@@H](O)[...,1,-2.3618,5.578099,88.3588,66.801411,6,0,1,9,...,0.007298,-4.697970,463.869102,14.056639,20.260688,149.15,2.397895,7.686143,2850.0,-0.155
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9579,C1=C(Cl)C(=C(C2=C1NC(=O)C(N2)=O)[N+](=O)[O-])Cl,0,0.1909,0.036443,57.1443,26.948379,6,0,3,4,...,0.003040,-5.036402,255.864815,15.050871,9.425904,101.34,0.000000,13.747503,460.0,2.533
9580,[C@H]3([N]2C1=C(C(=NC=N1)N)N=C2)[C@@H]([C@@H](...,0,-4.7530,22.591009,84.4263,54.579446,0,0,6,5,...,0.009701,-3.617143,389.163321,14.413456,16.102798,182.18,3.044522,8.125250,1993.0,-4.423
9581,[O+]1=N[N](C=C1[N-]C(NC2=CC=CC=C2)=O)C(CC3=CC=...,0,1.4656,2.147983,98.9391,49.686274,0,0,4,2,...,0.001331,-8.264830,440.311114,18.346296,14.609956,44.70,2.397895,7.670071,1597.0,3.090
9582,C1=C(OC)C(=CC2=C1C(=[N+](C(=C2CC)C)[NH-])C3=CC...,0,-0.8870,0.786769,113.2376,61.464618,0,0,2,4,...,0.005528,-5.223972,454.650380,16.237514,17.134680,39.93,0.000000,7.077579,1854.0,3.247


In [32]:
print(X_final["Original_SMILES"].isna().sum())

0


In [33]:
def create_stratified_splits(X, y, splits_config):
    """Create multiple stratified train-test splits without data leakage"""
    
    print("="*70)
    print("STEP 5: CREATE STRATIFIED TRAIN-TEST SPLITS")
    print("="*70)
    
    results = {}
    
    for split_name, test_size, seed in splits_config:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, 
            test_size=test_size,
            random_state=seed,
            stratify=y
        )
        
        results[split_name] = {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test
        }
        
        train_ratio = f"{int((1-test_size)*100)}/{int(test_size*100)}"
        print(f"{split_name} ({train_ratio}):")
        print(f"  Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
        print(f"  Train class dist: {y_train.value_counts().to_dict()}")
        print(f"  Test class dist: {y_test.value_counts().to_dict()}")
    
    print()
    return results

# Define split configurations: (name, test_size, random_seed)
splits_config = [
    ('Trial_70_30', 0.30, 42),
    ('Trial_80_20', 0.20, 42),
    ('Trial_60_40', 0.40, 42)
]

split_data = create_stratified_splits(X_final, y_clean, splits_config) #with the cleaned data

STEP 5: CREATE STRATIFIED TRAIN-TEST SPLITS
Trial_70_30 (70/30):
  Train: 6624 samples, Test: 2839 samples
  Train class dist: {1: 4700, 0: 1924}
  Test class dist: {1: 2015, 0: 824}
Trial_80_20 (80/20):
  Train: 7570 samples, Test: 1893 samples
  Train class dist: {1: 5372, 0: 2198}
  Test class dist: {1: 1343, 0: 550}
Trial_60_40 (60/40):
  Train: 5677 samples, Test: 3786 samples
  Train class dist: {1: 4028, 0: 1649}
  Test class dist: {1: 2687, 0: 1099}



In [23]:
print(X_final.isna().isna().sum())
    

Original_Name      0
Original_SMILES    0
nAcid              0
ALogP              0
ALogp2             0
                  ..
TopoPSA            0
SRW5               0
AMW                0
WPATH              0
XLogP              0
Length: 935, dtype: int64


In [21]:
print("="*70)
print("SPLIT DATA STRUCTURE AND INSPECTION")
print("="*70)

print(f"\nDict keys: {split_data.keys()}\n")

for trial_name, trial_data in split_data.items():
    print(f"{trial_name}:")
    print(f"  X_train shape: {trial_data['X_train'].shape}")
    print(f"  X_test shape: {trial_data['X_test'].shape}")
    print(f"  y_train shape: {trial_data['y_train'].shape}")
    print(f"  y_test shape: {trial_data['y_test'].shape}")
    print(f"\n  X_train head:\n{trial_data['X_train'].head()}")
    print(f"\n  y_train head:\n{trial_data['y_train'].head()}\n")
    print("-"*70)


SPLIT DATA STRUCTURE AND INSPECTION

Dict keys: dict_keys(['Trial_70_30', 'Trial_80_20', 'Trial_60_40'])

Trial_70_30:
  X_train shape: (6624, 935)
  X_test shape: (2839, 935)
  y_train shape: (6624,)
  y_test shape: (2839,)

  X_train head:
           Original_Name                                    Original_SMILES  \
666          loreclezole                     Cl/C(=C\n1cncn1)c1ccc(Cl)cc1Cl   
2033          fluotracen  C[C@H]1c2ccccc2[C@@H](CCCN(C)C)c2cc(C(F)(F)F)c...   
7444        zinc95093058  Cc1oc(=O)oc1COC(=O)[C@H]1N2C(=O)[C@@H](NC(=O)[...   
2546  (s)-allomethadione                          C=CCN1C(=O)O[C@@H](C)C1=O   
784       dihydrocodeine  COc1ccc2c3c1O[C@H]1[C@@H](O)CC[C@H]4[C@@H](C2)...   

      nAcid   ALogP    ALogp2      AMR       apol  naAromAtom  nB  nN  ...  \
666       0  1.8173  3.302579  28.4057  31.440758          11   0   3  ...   
2033      0  1.5694  2.463016  45.8867  55.734032          12   0   1  ...   
7444      0 -0.3542  0.125458  78.1919  64.110239

In [34]:
print("="*70)
print("MISSING VALUES CHECK ACROSS ALL SPLITS")
print("="*70)

for trial_name, trial_data in split_data.items():
    X_train = trial_data['X_train']
    X_test = trial_data['X_test']
    
    train_missing_count = X_train.isnull().sum().sum()
    test_missing_count = X_test.isnull().sum().sum()
    
    train_rows_with_missing = (X_train.isnull().sum(axis=1) > 0).sum()
    test_rows_with_missing = (X_test.isnull().sum(axis=1) > 0).sum()
    
    print(f"\n{trial_name}:")
    print(f"  Train - Total missing values: {train_missing_count}, Rows with missing: {train_rows_with_missing}")
    print(f"  Test  - Total missing values: {test_missing_count}, Rows with missing: {test_rows_with_missing}")

print("\n" + "="*70)


MISSING VALUES CHECK ACROSS ALL SPLITS

Trial_70_30:
  Train - Total missing values: 0, Rows with missing: 0
  Test  - Total missing values: 0, Rows with missing: 0

Trial_80_20:
  Train - Total missing values: 0, Rows with missing: 0
  Test  - Total missing values: 0, Rows with missing: 0

Trial_60_40:
  Train - Total missing values: 0, Rows with missing: 0
  Test  - Total missing values: 0, Rows with missing: 0



In [35]:
"""
loaded og data - removed descriptors with high missingness - removed descriptors with high correlation - removed molecules with high missingness
last step - removed all molecules missing values (1.3% of data so no big loss)
final size - 
9463 molecules × 933 descriptors
"""


'\nloaded og data - removed descriptors with high missingness - removed descriptors with high correlation - removed molecules with high missingness\nlast step - removed all molecules missing values (1.3% of data so no big loss)\nfinal size - \n9463 molecules × 933 descriptors\n'

### preparing data for model training


In [37]:
from pathlib import Path
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# EXPORT SPLITS TO data/split/<ratio>/{x_train,y_train,x_test,y_test}.csv
# -----------------------------------------------------------------------------

def _ratio_folder_from_split_name(split_name: str) -> str:
    # Expected: Trial_70_30 / Trial_80_20 / Trial_60_40
    parts = split_name.split('_')
    if len(parts) >= 3 and parts[-2].isdigit() and parts[-1].isdigit():
        return f"{parts[-2]}{parts[-1]}"
    # Fallback: keep name safe
    return split_name.replace('/', '_')


def export_split_data_to_csv(split_data: dict, base_dir: str = "../data/split") -> None:
    base_path = Path(base_dir)
    base_path.mkdir(parents=True, exist_ok=True)

    print("=" * 70)
    print(f"EXPORTING SPLITS TO: {base_path.resolve()}")
    print("=" * 70)

    for split_name, trial_data in split_data.items():
        ratio_folder = _ratio_folder_from_split_name(split_name)
        out_dir = base_path / ratio_folder
        out_dir.mkdir(parents=True, exist_ok=True)

        X_train = trial_data['X_train']
        y_train = trial_data['y_train']
        X_test = trial_data['X_test']
        y_test = trial_data['y_test']

        # Persist with indices to preserve alignment/debuggability
        X_train.to_csv(out_dir / "x_train.csv", index=True)
        y_train.to_csv(out_dir / "y_train.csv", index=True, header=True)
        X_test.to_csv(out_dir / "x_test.csv", index=True)
        y_test.to_csv(out_dir / "y_test.csv", index=True, header=True)

        print(f"{split_name} -> {out_dir}")
        print(f"  x_train: {X_train.shape} | y_train: {y_train.shape}")
        print(f"  x_test : {X_test.shape} | y_test : {y_test.shape}")

    print("\nDone.")


# Run export
export_split_data_to_csv(split_data, base_dir="../data/split")

EXPORTING SPLITS TO: /Users/soham/Desktop/afr_project/mtoralzml/data/split
Trial_70_30 -> ../data/split/7030
  x_train: (6624, 934) | y_train: (6624,)
  x_test : (2839, 934) | y_test : (2839,)
Trial_80_20 -> ../data/split/8020
  x_train: (7570, 934) | y_train: (7570,)
  x_test : (1893, 934) | y_test : (1893,)
Trial_60_40 -> ../data/split/6040
  x_train: (5677, 934) | y_train: (5677,)
  x_test : (3786, 934) | y_test : (3786,)

Done.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, matthews_corrcoef
 )

# Sampling methods
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

# -----------------------------------------------------------------------------
# COMPARE SAMPLING STRATEGIES ON 70/30 SPLIT ONLY
# -----------------------------------------------------------------------------

split_key = 'Trial_70_30'
if split_key not in split_data:
    raise KeyError(f"Expected '{split_key}' in split_data. Found: {list(split_data.keys())}")

trial = split_data[split_key]
X_train_full = trial['X_train']
y_train = trial['y_train']
X_test_full = trial['X_test']
y_test = trial['y_test']

# Keep only numeric descriptors for modeling (drop e.g. SMILES string columns)
X_train = X_train_full.select_dtypes(include=[np.number])
X_test = X_test_full.select_dtypes(include=[np.number])

print("=" * 70)
print("SAMPLING STRATEGY COMPARISON (70/30 split)")
print("=" * 70)
print(f"Train shape: {X_train.shape} | Test shape: {X_test.shape}")
print(f"Train class dist: {y_train.value_counts().to_dict()}")
print(f"Test class dist : {y_test.value_counts().to_dict()}")

base_model = Pipeline([
    ('scaler', StandardScaler(with_mean=True, with_std=True)),
    ('clf', LogisticRegression(max_iter=5000, solver='lbfgs', n_jobs=None, random_state=42)),
])

samplers = {
    'none': None,
    'random_over': RandomOverSampler(random_state=42),
    'smote': SMOTE(random_state=42, k_neighbors=5),
    'adasyn': ADASYN(random_state=42),
    'random_under': RandomUnderSampler(random_state=42),
    'smoteenn': SMOTEENN(random_state=42),
    'smotetomek': SMOTETomek(random_state=42),
}

rows = []
for sampler_name, sampler in samplers.items():
    if sampler is None:
        X_res, y_res = X_train, y_train
    else:
        X_res, y_res = sampler.fit_resample(X_train, y_train)

    model = base_model
    model.fit(X_res, y_res)
    y_pred = model.predict(X_test)

    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = None

    cm = confusion_matrix(y_test, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
    else:
        specificity = np.nan

    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan
    pr_auc = average_precision_score(y_test, y_proba) if y_proba is not None else np.nan

    rows.append({
        'split': split_key,
        'sampler': sampler_name,
        'train_n': int(X_res.shape[0]),
        'train_pos': int(np.sum(y_res == 1)),
        'train_neg': int(np.sum(y_res == 0)),
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_test, y_pred),
        'specificity': specificity,
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
    })

results_7030 = pd.DataFrame(rows).sort_values(by=['balanced_accuracy', 'mcc', 'f1'], ascending=False)
display(results_7030)

# Optional export of the comparison table
out_path = Path("../output/models") / "sampling_strategies_7030_logreg.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
results_7030.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")